# 01 Polymarket API Exploration

This notebook tests whether Polymarket API data can be retrieved for temperature-based markets.

The initial target is to retrieve event metadata, outcome markets, outcome prices, condition IDs, CLOB token IDs, and price history for at least one outcome token.

This is a feasibility notebook, not the final data pipeline.

In [2]:
import requests
import pandas as pd
import json
from datetime import datetime, timezone, timedelta
from pprint import pprint

GAMMA_BASE = "https://gamma-api.polymarket.com"
CLOB_BASE = "https://clob.polymarket.com"

def get_json(url, params=None, timeout=20):
    response = requests.get(url, params=params, timeout=timeout)
    print("URL:", response.url)
    print("Status code:", response.status_code)
    response.raise_for_status()
    return response.json()

def pretty(obj, max_chars=3000):
    text = json.dumps(obj, indent=2, ensure_ascii=False)
    print(text[:max_chars])
    if len(text) > max_chars:
        print(f"\n... truncated, total length = {len(text)} characters")

## 1. Choose a temperature-market event

The slug is taken from the Polymarket URL after `/event/`.

In [4]:
event_slug = "highest-temperature-in-hong-kong-on-june-10-2026"
event_slug

'highest-temperature-in-hong-kong-on-june-10-2026'

## 2. Fetch event metadata from Gamma API

In [6]:
event_data = get_json(
    f"{GAMMA_BASE}/events",
    params={"slug": event_slug}
)

type(event_data), len(event_data) if isinstance(event_data, list) else None

URL: https://gamma-api.polymarket.com/events?slug=highest-temperature-in-hong-kong-on-june-10-2026
Status code: 200


(list, 1)

In [7]:
pretty(event_data, max_chars=5000)

[
  {
    "id": "571434",
    "ticker": "highest-temperature-in-hong-kong-on-june-10-2026",
    "slug": "highest-temperature-in-hong-kong-on-june-10-2026",
    "title": "Highest temperature in Hong Kong on June 10?",
    "description": "This market will resolve to the temperature range that contains the highest temperature recorded by the Hong Kong Observatory in degrees Celsius on 10 Jun '26.\n\nThe resolution source for this market will be information from the Hong Kong Observatory, specifically the \"Absolute Daily Max (deg. C)\" the specified date once information is finalized in the relevant \"Daily Extract\", available here: https://www.weather.gov.hk/en/cis/climat.htm\n\nThis market can not resolve until data for this date has been published.\n\nThe resolution source for this market measures temperatures in Celsius to one decimal place (eg, 9.1°C). Thus, this is the level of precision that will be used when resolving the market.\n\nAny revisions to temperatures recorded after da

## 3. Normalise the event object

In [9]:
if isinstance(event_data, list):
    if len(event_data) == 0:
        raise ValueError("No event found for this slug.")
    event = event_data[0]
else:
    event = event_data

event_summary = {
    "id": event.get("id"),
    "ticker": event.get("ticker"),
    "slug": event.get("slug"),
    "title": event.get("title"),
    "description": event.get("description"),
    "resolutionSource": event.get("resolutionSource"),
    "startDate": event.get("startDate"),
    "endDate": event.get("endDate"),
    "active": event.get("active"),
    "closed": event.get("closed"),
    "archived": event.get("archived"),
    "volume": event.get("volume"),
    "liquidity": event.get("liquidity"),
}

pd.DataFrame([event_summary])

,id,ticker,slug,title,description,resolutionSource,startDate,endDate,active,closed,archived,volume,liquidity
0,571434,highest-temperature-in-hong-kong-on-june-10-2026,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,This market will resolve to the temperature ra...,None,2026-06-08T04:26:35.953385Z,2026-06-10T12:00:00Z,True,True,False,344382.890575,None


## 4. Extract associated outcome markets

In [11]:
markets = event.get("markets", [])
print(f"Number of markets inside event: {len(markets)}")

if markets:
    print("Example market keys:")
    print(sorted(markets[0].keys()))
else:
    print("No markets found inside event object.")

Number of markets inside event: 11
Example market keys:
['acceptingOrders', 'acceptingOrdersTimestamp', 'active', 'approved', 'archived', 'automaticallyActive', 'automaticallyResolved', 'bestAsk', 'clearBookOnStart', 'clobTokenIds', 'closed', 'closedTime', 'competitive', 'conditionId', 'createdAt', 'customLiveness', 'cyom', 'deploying', 'deployingTimestamp', 'description', 'enableOrderBook', 'endDate', 'endDateIso', 'featured', 'feeSchedule', 'feeType', 'feesEnabled', 'funded', 'gameStartTime', 'groupItemThreshold', 'groupItemTitle', 'hasReviewedDates', 'holdingRewardsEnabled', 'icon', 'id', 'image', 'lastTradePrice', 'liquidity', 'liquidityAmm', 'liquidityClob', 'liquidityNum', 'makerBaseFee', 'manualActivation', 'marketMakerAddress', 'negRisk', 'negRiskMarketID', 'negRiskOther', 'negRiskRequestID', 'new', 'orderMinSize', 'orderPriceMinTickSize', 'outcomePrices', 'outcomes', 'pagerDutyNotificationEnabled', 'pendingDeployment', 'question', 'questionID', 'ready', 'requiresTranslation', 

In [12]:
rows = []

for m in markets:
    rows.append({
        "market_id": m.get("id"),
        "question": m.get("question"),
        "slug": m.get("slug"),
        "condition_id": m.get("conditionId"),
        "resolution_source": m.get("resolutionSource"),
        "end_date": m.get("endDate"),
        "active": m.get("active"),
        "closed": m.get("closed"),
        "volume": m.get("volume"),
        "liquidity": m.get("liquidity"),
        "outcomes_raw": m.get("outcomes"),
        "outcome_prices_raw": m.get("outcomePrices"),
        "clob_token_ids_raw": m.get("clobTokenIds") or m.get("clob_token_ids"),
    })

market_df = pd.DataFrame(rows)
market_df

,market_id,question,slug,condition_id,resolution_source,end_date,active,closed,volume,liquidity,outcomes_raw,outcome_prices_raw,clob_token_ids_raw
0,2467879,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0x5403a3f05252b143a2bcca593e311f90b46701072ddc...,None,2026-06-10T12:00:00Z,True,True,2459.0020000000004,0,"[""Yes"", ""No""]","[""0"", ""1""]","[""75114801924149318393325176600531133058125419..."
1,2467880,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0xfbba10e711c534967a1609f2ec0dea4ef0c404327799...,None,2026-06-10T12:00:00Z,True,True,4339.706145,0,"[""Yes"", ""No""]","[""0"", ""1""]","[""71382408483524696020927376515786823240288329..."
2,2467881,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0x839c27f84d9ad85f82969125592c363afc30328dde45...,None,2026-06-10T12:00:00Z,True,True,14542.874648999994,None,"[""Yes"", ""No""]","[""0"", ""1""]","[""62422821600409057459698301447235282888890577..."
3,2467882,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0x1546a51a1f1bcf3cb7cccbd327a6abbf2408e4811bb3...,None,2026-06-10T12:00:00Z,True,True,39412.588582000026,None,"[""Yes"", ""No""]","[""0"", ""1""]","[""15718641776411360983478895165289796527836971..."
4,2467883,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0xea51e07e55f4106df178e7ca4a0ae90eda7d4a0f772f...,None,2026-06-10T12:00:00Z,True,True,79509.85583500004,None,"[""Yes"", ""No""]","[""1"", ""0""]","[""10395260835846391499998553181890750326538322..."
5,2467884,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0xe4ce7dcda61f5c04ce03c485d1e2a48d5ad332d78c49...,None,2026-06-10T12:00:00Z,True,True,42174.46050599999,None,"[""Yes"", ""No""]","[""0"", ""1""]","[""16397666266689147390934382558167088774462847..."
6,2467885,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0x1329f2bec82131fae4cb9c5ee30165cf0e6a6c652530...,None,2026-06-10T12:00:00Z,True,True,52907.91220299997,None,"[""Yes"", ""No""]","[""0"", ""1""]","[""60695725697130387319209925704743130722245093..."
7,2467886,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0x514c6d354a55c98201b4f365bc683bc4e19c921df980...,None,2026-06-10T12:00:00Z,True,True,46021.93409500009,None,"[""Yes"", ""No""]","[""0"", ""1""]","[""40310838860668352533543531372481493817226770..."
8,2467887,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0x030c0ce594a6ea1f944b14c3692d03d9e69cc21cd303...,None,2026-06-10T12:00:00Z,True,True,40645.79464400003,None,"[""Yes"", ""No""]","[""0"", ""1""]","[""32060790227972710091841899429607160109976959..."
9,2467888,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0xe072cb5dc1878f23c62faa54221cffb421d102a93762...,None,2026-06-10T12:00:00Z,True,True,14856.569109000007,None,"[""Yes"", ""No""]","[""0"", ""1""]","[""18042873604242480997582327441850239974300322..."


## 5. Convert market-level data into outcome-level table

In [14]:
def parse_maybe_json(x):
    if x is None:
        return None
    if isinstance(x, (list, dict)):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return x
    return x

long_rows = []

for m in markets:
    outcomes = parse_maybe_json(m.get("outcomes")) or []
    prices = parse_maybe_json(m.get("outcomePrices")) or []
    token_ids = parse_maybe_json(m.get("clobTokenIds") or m.get("clob_token_ids")) or []
    
    for i, outcome in enumerate(outcomes):
        long_rows.append({
            "event_slug": event_slug,
            "event_title": event.get("title"),
            "market_id": m.get("id"),
            "market_question": m.get("question"),
            "market_slug": m.get("slug"),
            "condition_id": m.get("conditionId"),
            "outcome_index": i,
            "outcome": outcome,
            "outcome_price": prices[i] if i < len(prices) else None,
            "clob_token_id": token_ids[i] if i < len(token_ids) else None,
            "active": m.get("active"),
            "closed": m.get("closed"),
            "volume": m.get("volume"),
            "liquidity": m.get("liquidity"),
        })

outcome_df = pd.DataFrame(long_rows)
outcome_df["outcome_price"] = pd.to_numeric(outcome_df["outcome_price"], errors="coerce")
outcome_df.sort_values("outcome_price", ascending=False)

,event_slug,event_title,market_id,market_question,market_slug,condition_id,outcome_index,outcome,outcome_price,clob_token_id,active,closed,volume,liquidity
11,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467884,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0xe4ce7dcda61f5c04ce03c485d1e2a48d5ad332d78c49...,1,No,1,5857491092768695076259279133690879993503943059...,True,True,42174.46050599999,None
8,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467883,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0xea51e07e55f4106df178e7ca4a0ae90eda7d4a0f772f...,0,Yes,1,1039526083584639149999855318189075032653832204...,True,True,79509.85583500004,None
19,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467888,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0xe072cb5dc1878f23c62faa54221cffb421d102a93762...,1,No,1,4661695803079584919390531866781801790323011186...,True,True,14856.569109000007,None
17,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467887,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0x030c0ce594a6ea1f944b14c3692d03d9e69cc21cd303...,1,No,1,2363233767285758740849740154862542519524279020...,True,True,40645.79464400003,None
15,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467886,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0x514c6d354a55c98201b4f365bc683bc4e19c921df980...,1,No,1,3667361803328536724802078239555484475230126256...,True,True,46021.93409500009,None
13,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467885,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0x1329f2bec82131fae4cb9c5ee30165cf0e6a6c652530...,1,No,1,1132551791448792847694823024888203284241546344...,True,True,52907.91220299997,None
1,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467879,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0x5403a3f05252b143a2bcca593e311f90b46701072ddc...,1,No,1,1130452331264548567773751920519419890404900978...,True,True,2459.0020000000004,0
21,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467889,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0x4e91e59c0b253a405f045d69fe4a6c4a7f8763ad2898...,1,No,1,5279784164846580375880749104004243533339251603...,True,True,7512.192807,None
7,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467882,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0x1546a51a1f1bcf3cb7cccbd327a6abbf2408e4811bb3...,1,No,1,5444289154900069681984255671317744894261994507...,True,True,39412.588582000026,None
5,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467881,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0x839c27f84d9ad85f82969125592c363afc30328dde45...,1,No,1,6892450893976607252480640600448993236622650657...,True,True,14542.874648999994,None


## 6. Build YES-only probabilities and retrieve CLOB price history

In [16]:
# For temperature-bin events, each Polymarket sub-market is binary: Yes / No.
# The model-relevant probability for each bin is usually the YES price.

yes_df = outcome_df[outcome_df["outcome"].str.lower() == "yes"].copy()

yes_df = yes_df[[
    "event_slug",
    "event_title",
    "market_id",
    "market_question",
    "market_slug",
    "condition_id",
    "outcome",
    "outcome_price",
    "clob_token_id",
    "active",
    "closed",
    "volume",
    "liquidity",
]].sort_values("outcome_price", ascending=False)

yes_df

,event_slug,event_title,market_id,market_question,market_slug,condition_id,outcome,outcome_price,clob_token_id,active,closed,volume,liquidity
8,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467883,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0xea51e07e55f4106df178e7ca4a0ae90eda7d4a0f772f...,Yes,1,1039526083584639149999855318189075032653832204...,True,True,79509.85583500004,None
0,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467879,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0x5403a3f05252b143a2bcca593e311f90b46701072ddc...,Yes,0,7511480192414931839332517660053113305812541935...,True,True,2459.0020000000004,0
2,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467880,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0xfbba10e711c534967a1609f2ec0dea4ef0c404327799...,Yes,0,7138240848352469602092737651578682324028832907...,True,True,4339.706145,0
4,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467881,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0x839c27f84d9ad85f82969125592c363afc30328dde45...,Yes,0,6242282160040905745969830144723528288889057744...,True,True,14542.874648999994,None
6,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467882,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0x1546a51a1f1bcf3cb7cccbd327a6abbf2408e4811bb3...,Yes,0,1571864177641136098347889516528979652783697133...,True,True,39412.588582000026,None
10,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467884,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0xe4ce7dcda61f5c04ce03c485d1e2a48d5ad332d78c49...,Yes,0,1639766626668914739093438255816708877446284743...,True,True,42174.46050599999,None
12,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467885,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0x1329f2bec82131fae4cb9c5ee30165cf0e6a6c652530...,Yes,0,6069572569713038731920992570474313072224509346...,True,True,52907.91220299997,None
14,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467886,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0x514c6d354a55c98201b4f365bc683bc4e19c921df980...,Yes,0,4031083886066835253354353137248149381722677087...,True,True,46021.93409500009,None
16,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467887,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0x030c0ce594a6ea1f944b14c3692d03d9e69cc21cd303...,Yes,0,3206079022797271009184189942960716010997695920...,True,True,40645.79464400003,None
18,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467888,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-june-10-20...,0xe072cb5dc1878f23c62faa54221cffb421d102a93762...,Yes,0,1804287360424248099758232744185023997430032254...,True,True,14856.569109000007,None


In [17]:
# Basic sanity check: for a mutually exclusive temperature-bin event,
# the YES probabilities across all bins should roughly sum to 1,
# allowing for bid-ask spread, stale prices, market closure, and rounding.

yes_prob_sum = yes_df["outcome_price"].sum()

print("Number of YES bin markets:", len(yes_df))
print("Sum of YES outcome prices:", yes_prob_sum)

yes_df[["market_question", "outcome_price", "clob_token_id", "closed", "volume"]]

Number of YES bin markets: 11
Sum of YES outcome prices: 1


,market_question,outcome_price,clob_token_id,closed,volume
8,Will the highest temperature in Hong Kong be 2...,1,1039526083584639149999855318189075032653832204...,True,79509.85583500004
0,Will the highest temperature in Hong Kong be 2...,0,7511480192414931839332517660053113305812541935...,True,2459.0020000000004
2,Will the highest temperature in Hong Kong be 2...,0,7138240848352469602092737651578682324028832907...,True,4339.706145
4,Will the highest temperature in Hong Kong be 2...,0,6242282160040905745969830144723528288889057744...,True,14542.874648999994
6,Will the highest temperature in Hong Kong be 2...,0,1571864177641136098347889516528979652783697133...,True,39412.588582000026
10,Will the highest temperature in Hong Kong be 2...,0,1639766626668914739093438255816708877446284743...,True,42174.46050599999
12,Will the highest temperature in Hong Kong be 3...,0,6069572569713038731920992570474313072224509346...,True,52907.91220299997
14,Will the highest temperature in Hong Kong be 3...,0,4031083886066835253354353137248149381722677087...,True,46021.93409500009
16,Will the highest temperature in Hong Kong be 3...,0,3206079022797271009184189942960716010997695920...,True,40645.79464400003
18,Will the highest temperature in Hong Kong be 3...,0,1804287360424248099758232744185023997430032254...,True,14856.569109000007


In [18]:
# Choose a YES token rather than accidentally selecting a high-priced NO token.
# Prefer non-closed markets if available; otherwise use the highest-probability YES token.

if len(yes_df[yes_df["closed"] == False]) > 0:
    candidate = yes_df[yes_df["closed"] == False].sort_values("outcome_price", ascending=False).head(1)
else:
    candidate = yes_df.sort_values("outcome_price", ascending=False).head(1)

candidate

,event_slug,event_title,market_id,market_question,market_slug,condition_id,outcome,outcome_price,clob_token_id,active,closed,volume,liquidity
8,highest-temperature-in-hong-kong-on-june-10-2026,Highest temperature in Hong Kong on June 10?,2467883,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-june-10-20...,0xea51e07e55f4106df178e7ca4a0ae90eda7d4a0f772f...,Yes,1,1039526083584639149999855318189075032653832204...,True,True,79509.85583500004,None


In [19]:
# Test price history for the selected YES token.
# We first try interval="max" because it is the least restrictive query.

if len(candidate) == 0:
    print("No YES CLOB token ID found.")
else:
    token_id = str(candidate.iloc[0]["clob_token_id"])
    print("Testing price history for YES token_id:", token_id)
    
    try:
        price_history = get_json(
            f"{CLOB_BASE}/prices-history",
            params={
                "market": token_id,
                "interval": "max",
            }
        )
        pretty(price_history, max_chars=3000)
    except requests.HTTPError as e:
        print("HTTP error:", e)
        print("Response body:", e.response.text[:1000])

Testing price history for YES token_id: 103952608358463914999985531818907503265383220402407534332311099994541518367084
URL: https://clob.polymarket.com/prices-history?market=103952608358463914999985531818907503265383220402407534332311099994541518367084&interval=max
Status code: 200
{
  "history": [
    {
      "t": 1780891805,
      "p": 0.195
    },
    {
      "t": 1780892404,
      "p": 0.345
    },
    {
      "t": 1780893006,
      "p": 0.345
    },
    {
      "t": 1780893604,
      "p": 0.345
    },
    {
      "t": 1780894206,
      "p": 0.33
    },
    {
      "t": 1780894803,
      "p": 0.33
    },
    {
      "t": 1780895404,
      "p": 0.325
    },
    {
      "t": 1780896004,
      "p": 0.325
    },
    {
      "t": 1780896604,
      "p": 0.325
    },
    {
      "t": 1780897203,
      "p": 0.325
    },
    {
      "t": 1780897819,
      "p": 0.325
    },
    {
      "t": 1780898404,
      "p": 0.325
    },
    {
      "t": 1780899018,
      "p": 0.325
    },
    {
      "

In [20]:
if "price_history" in globals() and isinstance(price_history, dict) and "history" in price_history:
    hist_df = pd.DataFrame(price_history["history"])
    
    if len(hist_df) > 0:
        hist_df["datetime_utc"] = pd.to_datetime(hist_df["t"], unit="s", utc=True)
        hist_df["p"] = pd.to_numeric(hist_df["p"], errors="coerce")
        display(hist_df.head())
        display(hist_df.tail())
    else:
        print("Price history returned but is empty.")
else:
    print("No usable price history object.")

,t,p,datetime_utc
0,1780891805,0.195,2026-06-08 04:10:05+00:00
1,1780892404,0.345,2026-06-08 04:20:04+00:00
2,1780893006,0.345,2026-06-08 04:30:06+00:00
3,1780893604,0.345,2026-06-08 04:40:04+00:00
4,1780894206,0.330,2026-06-08 04:50:06+00:00


,t,p,datetime_utc
419,1781144404,0.9995,2026-06-11 02:20:04+00:00
420,1781145004,0.9995,2026-06-11 02:30:04+00:00
421,1781145603,0.9995,2026-06-11 02:40:03+00:00
422,1781146203,0.9995,2026-06-11 02:50:03+00:00
423,1781146806,0.9995,2026-06-11 03:00:06+00:00


In [21]:
# Try price history for all YES CLOB token IDs using interval="max".
# This checks whether the endpoint works for any token without restrictive timestamp filters.

history_results = []

for _, row in yes_df.dropna(subset=["clob_token_id"]).iterrows():
    token_id = str(row["clob_token_id"])
    
    try:
        ph = get_json(
            f"{CLOB_BASE}/prices-history",
            params={
                "market": token_id,
                "interval": "max",
            }
        )
        
        n_obs = len(ph.get("history", [])) if isinstance(ph, dict) else 0
        
        history_results.append({
            "market_question": row["market_question"],
            "outcome_price": row["outcome_price"],
            "clob_token_id": token_id,
            "n_history_observations": n_obs,
            "error": None,
        })
        
    except requests.HTTPError as e:
        history_results.append({
            "market_question": row["market_question"],
            "outcome_price": row["outcome_price"],
            "clob_token_id": token_id,
            "n_history_observations": None,
            "error": e.response.text[:300],
        })

history_check_df = pd.DataFrame(history_results)
history_check_df.sort_values("n_history_observations", ascending=False)

URL: https://clob.polymarket.com/prices-history?market=103952608358463914999985531818907503265383220402407534332311099994541518367084&interval=max
Status code: 200
URL: https://clob.polymarket.com/prices-history?market=75114801924149318393325176600531133058125419354659304337062518011403869315671&interval=max
Status code: 200
URL: https://clob.polymarket.com/prices-history?market=71382408483524696020927376515786823240288329073616677127679194797612102154666&interval=max
Status code: 200
URL: https://clob.polymarket.com/prices-history?market=62422821600409057459698301447235282888890577446869542091777011845356809868287&interval=max
Status code: 200
URL: https://clob.polymarket.com/prices-history?market=15718641776411360983478895165289796527836971336383880138777981093993104520784&interval=max
Status code: 200
URL: https://clob.polymarket.com/prices-history?market=16397666266689147390934382558167088774462847438957547963956284647658172511927&interval=max
Status code: 200
URL: https://clob.pol

,market_question,outcome_price,clob_token_id,n_history_observations,error
0,Will the highest temperature in Hong Kong be 2...,1,1039526083584639149999855318189075032653832204...,424,None
1,Will the highest temperature in Hong Kong be 2...,0,7511480192414931839332517660053113305812541935...,424,None
2,Will the highest temperature in Hong Kong be 2...,0,7138240848352469602092737651578682324028832907...,424,None
5,Will the highest temperature in Hong Kong be 2...,0,1639766626668914739093438255816708877446284743...,424,None
6,Will the highest temperature in Hong Kong be 3...,0,6069572569713038731920992570474313072224509346...,424,None
7,Will the highest temperature in Hong Kong be 3...,0,4031083886066835253354353137248149381722677087...,424,None
8,Will the highest temperature in Hong Kong be 3...,0,3206079022797271009184189942960716010997695920...,424,None
9,Will the highest temperature in Hong Kong be 3...,0,1804287360424248099758232744185023997430032254...,424,None
10,Will the highest temperature in Hong Kong be 3...,0,8840898554643784716059101654562265380799762329...,424,None
3,Will the highest temperature in Hong Kong be 2...,0,6242282160040905745969830144723528288889057744...,423,None


In [22]:
# Check current best bid/ask-style price for the selected YES token.
# This helps verify that the CLOB token ID is usable even if price history is empty.

if len(candidate) > 0:
    token_id = str(candidate.iloc[0]["clob_token_id"])
    
    for side in ["BUY", "SELL"]:
        try:
            current_price = get_json(
                f"{CLOB_BASE}/price",
                params={
                    "token_id": token_id,
                    "side": side,
                }
            )
            print(side, current_price)
        except requests.HTTPError as e:
            print(side, "HTTP error:", e)
            print("Response body:", e.response.text[:500])

URL: https://clob.polymarket.com/price?token_id=103952608358463914999985531818907503265383220402407534332311099994541518367084&side=BUY
Status code: 404
BUY HTTP error: 404 Client Error: Not Found for url: https://clob.polymarket.com/price?token_id=103952608358463914999985531818907503265383220402407534332311099994541518367084&side=BUY
Response body: {"error":"No orderbook exists for the requested token id"}

URL: https://clob.polymarket.com/price?token_id=103952608358463914999985531818907503265383220402407534332311099994541518367084&side=SELL
Status code: 404
SELL HTTP error: 404 Client Error: Not Found for url: https://clob.polymarket.com/price?token_id=103952608358463914999985531818907503265383220402407534332311099994541518367084&side=SELL
Response body: {"error":"No orderbook exists for the requested token id"}



## 7. Save lightweight local outputs

In [24]:
import os

os.makedirs("../data/raw/polymarket", exist_ok=True)
os.makedirs("../data/processed/polymarket", exist_ok=True)

with open("../data/raw/polymarket/sample_event_response.json", "w") as f:
    json.dump(event_data, f, indent=2)

outcome_df.to_csv("../data/processed/polymarket/sample_outcome_table.csv", index=False)
yes_df.to_csv("../data/processed/polymarket/sample_yes_probability_table.csv", index=False)

if "hist_df" in globals() and len(hist_df) > 0:
    hist_df.to_csv("../data/processed/polymarket/sample_price_history.csv", index=False)

print("Saved local API exploration outputs. These files are ignored by git.")

Saved local API exploration outputs. These files are ignored by git.


## Priority 5 findings

This notebook checks whether Polymarket API data can be retrieved for a real temperature event.

Important fields for the dissertation are event title, market question, outcome labels, market-implied outcome prices, condition ID, CLOB token IDs and price history availability.

Next steps:

1. Repeat this for London and NYC markets.
2. Build a reusable function that takes an event slug and returns a clean outcome-level table.
3. Join the API output with the manually curated market-universe table.
4. Use token IDs to retrieve historical prices.
5. Align price timestamps with forecast issue times and event resolution times.

## Current API result

The Gamma API successfully retrieved event-level metadata for a real Polymarket temperature event.

For the Hong Kong June 10 event, the API returned:

- event ID;
- event title;
- event description and resolution rule;
- 11 associated binary outcome markets;
- market IDs;
- condition IDs;
- outcome labels;
- outcome prices;
- CLOB token IDs;
- market volumes and closed/active flags.

A clean YES-only table was created. This is the relevant table for interpreting the market as an implied probability distribution over temperature bins. In this example, the YES prices across the 11 bins sum to 1, which is consistent with the interpretation of the event as a multi-bin temperature distribution.

The CLOB price-history endpoint works when using `interval="max"`. For the selected YES token, the endpoint returned a non-empty historical price path, showing that historical market-implied probabilities can be retrieved.

The live `/price` endpoint returned `No orderbook exists` for the selected token, likely because the market is already closed/resolved. This is not a blocker for historical analysis, but active/future markets should be tested next for live order-book and bid-ask data.

For the dissertation, the key lesson is that a Polymarket temperature-bin event is represented as multiple binary YES/NO markets. Therefore, the model-implied predictive distribution should be compared against the YES prices for each bin.